### Imports

In [1]:
import os
import json
import chromadb
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
import hashlib
import shutil
from pathlib import Path

In [2]:
# Getting API Key
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import torch
print(torch.cuda.is_available())

True


In [4]:
# ChromaDB path setup
CHROMA_PATH = "./chroma_db"
COLLECTION = "TTRPG_corpus"
HASH_FILE = "./chroma_db/corpus.hash"
SOURCES_FILE = "./chroma_db/embedded_sources.json"

In [5]:
# Embedder
embedder = HuggingFaceEmbeddings(
    model_name = "BAAI/bge-m3",
    model_kwargs = {"device": "cuda"},
    encode_kwargs = {"normalize_embeddings": True}
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [6]:
embedder._client.device

device(type='cuda', index=0)

### Static Corpus

In [8]:
# Get the corpus chunks.
with open('corpus_chunks.json', 'r', encoding='utf-8') as f:
    raw_chunks = json.load(f)

all_chunks = [
    Document(
        page_content=chunk["text"],
        metadata=chunk['metadata']
    )
    for chunk in raw_chunks
]

In [9]:
# The Main Corpus, Permanent Embedding
# This will embed everything on first run to create the hash.

# Chroma has batching constraints.
BATCH_SIZE = 5000

# This reads corpus_chunks.json and hashes it.
def corpus_hash(filepath="corpus_chunks.json"):
    with open(filepath, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

# Saves the hash and also a list of the files that were embedded.
def save_state(current_hash, embedded_sources):
    open(HASH_FILE, "w").write(current_hash)
    json.dump(list(embedded_sources), open(SOURCES_FILE, "w"))

current_hash = corpus_hash()
stored_hash = open(HASH_FILE).read().strip() if Path(HASH_FILE).exists() else None
stored_sources = set(json.load(open(SOURCES_FILE))) if Path(SOURCES_FILE).exists() else set()
current_sources = set(c.metadata["source"] for c in all_chunks)
new_sources = current_sources - stored_sources

# If nothing has changed, the hash pulls from stored.
if current_hash == stored_hash:
    print("Corpus unchanged. Loading existing store.")
    corpus_store = Chroma(
        persist_directory=CHROMA_PATH,
        embedding_function=embedder,
        collection_name=COLLECTION
    )

# If only a source is added, then only the added source will be added.
elif new_sources and (current_sources - new_sources == stored_sources):
    print(f"New sources detected: {new_sources}. Embedding incrementally...")
    corpus_store = Chroma(
        persist_directory=CHROMA_PATH,
        embedding_function=embedder,
        collection_name=COLLECTION
    )
    new_chunks = [c for c in all_chunks if c.metadata["source"] in new_sources]
    for i in range(0, len(new_chunks), BATCH_SIZE):
        batch = new_chunks[i:i + BATCH_SIZE]
        corpus_store.add_documents(batch)
    save_state(current_hash, current_sources)
    print(f"Added {len(new_chunks)} chunks from {len(new_sources)} new source(s).")

# If a source has changed, the vector store will be deleted and everything will re-embed.
else:
    print("Corpus changed. Rebuilding from scratch...")
    if Path(CHROMA_PATH).exists():
        shutil.rmtree(CHROMA_PATH)
    corpus_store = Chroma(
        persist_directory=CHROMA_PATH,
        embedding_function=embedder,
        collection_name=COLLECTION
    )
    for i in range(0, len(all_chunks), BATCH_SIZE):
        batch = all_chunks[i:i + BATCH_SIZE]
        corpus_store.add_documents(batch)
    save_state(current_hash, current_sources)
    print("Full rebuild complete.")

New sources detected: {'dunsay_the_king_of_elflands_daughter', 'gonnerman_the_dark_temple', 'johnson_the_witch_tree', 'doyle_the_adventures_of_sherlock_holmes', 'gal_cairn_warden_book', 'dumas_the_Three_musketeers', 'boyle-associates_eclipse_phase', 'homer_the_odyssey', 'dickens_great_expectations', 'wister_the_virginian_a_horseman_of_the_plains', 'verne_twenty_thousand_leagues_under_the_sea', 'lovecraft_at_the_mountains_of_madness', 'defoe_the_life_and_adventures_of_robinson_crusoe', 'law_gumshoe_srd', 'jerome_three_men_in_a_boat', 'milne_winnie_the_pooh', 'many_one_page_dungeon_contest_2025_compendium', 'nilsson-nohr_mork_borg', 'harper_blades_in_the_dark', 'anonymouos_the_poetic_edda', 'wizards_dungeons_and_dragons_v5.2.1', 'neal_monkey_isle', 'malory_king_arthur_and_the_knights_of_the_round_table', 'anonymous_the_arabian_nights_entertainment', 'dostoyevsky_crime_and_punishment', 'tomkin_ironsworn_starforged', 'gonnerman_morgansfort', 'emily_bronte_wuthering_heights', 'tomkin_ironsw

Citation:

Brainstorming and refinement: 

Anthropic. (2026). Claude Sonnet 4.6 [AI language model]. https://claude.ai 